# Base Messages

The `base.py` module defines the shared message structure used by LangChain chat models. It provides base message classes, streaming message chunks, text extraction, content-block conversion, serialization helpers, and content-merging utilities.

# TextAccessor

`TextAccessor` is a string-like compatibility class that supports both property-style and legacy method-style access to message text.

## Bases

- `str`

### Methods

1. `__new__`: Creates a new `TextAccessor` instance from a string value.
   * **Syntax:**
     ```python
     __new__(
         cls,
         value: str # String value stored in the accessor
     ) -> Self
     ```

2. `__call__`: Returns the stored string when the accessor is called like a method.

   Calling message text through `.text()` is deprecated. Property access through `.text` should be used instead.

   * **Syntax:**
     ```python
     __call__(
         self
     ) -> str
     ```

# BaseMessage

`BaseMessage` is the common base class for messages exchanged with chat models. It stores message content, provider-specific data, response metadata, type information, an optional name, and an optional identifier.

## Bases

- `Serializable`

## Attributes

1. `content`: Stores the contents of the message as plain text or a list of text and dictionary blocks.
   * **Type:**
     ```python
     content: str | list[str | dict[Any, Any]]
     ```

2. `additional_kwargs`: Stores additional provider-specific payload data associated with the message.
   * **Type:**
     ```python
     additional_kwargs: dict[Any, Any] = Field(
         default_factory=dict
     )
     ```

3. `response_metadata`: Stores response information such as headers, log probabilities, token counts, or model names.
   * **Type:**
     ```python
     response_metadata: dict[Any, Any] = Field(
         default_factory=dict
     )
     ```

4. `type`: Stores the unique message type used during serialization and deserialization.
   * **Type:**
     ```python
     type: str
     ```

5. `name`: Stores an optional human-readable name for the message.
   * **Type:**
     ```python
     name: str | None = None
     ```

6. `id`: Stores an optional unique identifier for the message.

   Numeric identifiers are converted to strings automatically.

   * **Type:**
     ```python
     id: str | None = Field(
         default=None,
         coerce_numbers_to_str=True
     )
     ```

## Configuration

1. `model_config`: Allows additional fields that are not explicitly declared on the model.
   * **Definition:**
     ```python
     model_config = ConfigDict(
         extra="allow"
     )
     ```

### Properties

1. `content_blocks`: Converts the message content into standardized LangChain content blocks.

   Plain strings become text blocks. Known blocks are preserved, while provider-specific or older block formats are translated when possible and otherwise kept as non-standard blocks.

   * **Type:**
     ```python
     content_blocks: list[types.ContentBlock]
     ```

2. `text`: Returns only the textual content of the message as a `TextAccessor`.

   For list-based content, only strings and dictionaries with `type="text"` are included. Other content-block types are ignored.

   * **Type:**
     ```python
     text: TextAccessor
     ```

### Methods

1. `__init__`: Initializes a message from raw content or standardized content blocks.
   * **Syntax:**
     ```python
     __init__(
         self,
         content: str | list[str | dict[Any, Any]] | None = None, # Raw message content
         content_blocks: list[types.ContentBlock] | None = None, # Standardized content blocks
         **kwargs: Any # Additional message fields
     ) -> None
     ```

2. `is_lc_serializable`: Indicates that `BaseMessage` supports LangChain serialization.
   * **Syntax:**
     ```python
     @classmethod
     is_lc_serializable(
         cls
     ) -> bool
     ```

3. `get_lc_namespace`: Returns the LangChain serialization namespace used for messages.
   * **Syntax:**
     ```python
     @classmethod
     get_lc_namespace(
         cls
     ) -> list[str]
     ```

4. `__add__`: Combines the current message with another message-like object and returns a `ChatPromptTemplate`.
   * **Syntax:**
     ```python
     __add__(
         self,
         other: Any # Message or message-like object to concatenate
     ) -> ChatPromptTemplate
     ```

5. `pretty_repr`: Returns a readable representation of the message.

   The representation includes a formatted title, the optional message name, and the message content.

   * **Syntax:**
     ```python
     pretty_repr(
         self,
         html: bool = False # Whether to use HTML-style formatting
     ) -> str
     ```

6. `pretty_print`: Prints a readable representation of the message.
   * **Syntax:**
     ```python
     pretty_print(
         self
     ) -> None
     ```

# BaseMessageChunk

`BaseMessageChunk` represents a partial message produced during streaming. Message chunks can be combined to form a complete message.

## Bases

- `BaseMessage`

### Methods

1. `__add__`: Combines the current message chunk with another chunk or a list of chunks.

   It merges content, additional keyword arguments, and response metadata. A `TypeError` is raised when the supplied value is not a message chunk or a list of message chunks.

   * **Syntax:**
     ```python
     __add__(
         self,
         other: Any # Message chunk or list of message chunks to combine
     ) -> BaseMessageChunk
     ```

## Functions

1. `merge_content`: Merges multiple message-content values into one string or list.

   Strings are concatenated directly. Lists are merged, and a string may be appended to the final string element of an existing list.

   * **Syntax:**
     ```python
     merge_content(
         first_content: str | list[str | dict[Any, Any]], # First content value
         *contents: str | list[str | dict[Any, Any]] # Additional content values
     ) -> str | list[str | dict[Any, Any]]
     ```

2. `message_to_dict`: Converts one message into a dictionary containing its type and serialized data.
   * **Syntax:**
     ```python
     message_to_dict(
         message: BaseMessage # Message to serialize
     ) -> dict[str, Any]
     ```

3. `messages_to_dict`: Converts a sequence of messages into a list of serialized dictionaries.
   * **Syntax:**
     ```python
     messages_to_dict(
         messages: Sequence[BaseMessage] # Messages to serialize
     ) -> list[dict[str, Any]]
     ```

4. `get_msg_title_repr`: Creates a centered title surrounded by equals signs.

   The title can optionally be formatted in bold.

   * **Syntax:**
     ```python
     get_msg_title_repr(
         title: str, # Title to format
         *,
         bold: bool = False # Whether to make the title bold
     ) -> str
     ```